<a href="https://colab.research.google.com/github/xidoudou/ai-agents/blob/main/agent01_unit_converter_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install & Import

In [36]:
!pip install openai
import re, os
from openai import OpenAI

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

Tools & Functions

In [37]:
def km_to_miles(km):
  return float(km) * 0.621371
def miles_to_km(miles):
  return float(miles) * 1.60934
def celsius_to_fahrenheit(c):
  return float(c) * 9/5 + 32
def fahrenheit_to_celsius(f):
  return (float(f) - 32) * 5/9


known_actions= {
    "km_to_miles": km_to_miles,
    "miles_to_km": miles_to_km,
    "celsius_to_fahrenheit": celsius_to_fahrenheit,
    "fahrenheit_to_celsius": fahrenheit_to_celsius
}

System Prompt

In [38]:
prompt = """
You run in a loop of Thought, Action, Pause, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return Pause.
Observation will be the result of running thoes actions.

Your available actions are:
km_to_miles:
e.g. km_to_miles: 10
convert the input kilometer number into miles

miles_to_km:
e.g. miles_to_km: 3
convert the input miles number into kilometer

celsius_to_fahrenheit:
e.g. celsius_to_fahrenheit: 20
convert the input celsius temperature into fahrenheit

fahrenheit_to_celsius:
e.g. fahrenheit_to_celsius: 59
convert the input fahrenheit temperature into celsius


Example session:

Question: how many kilometers is 4 miles?
Thought: I should convert the input miles number into kilometer
Action: miles_to_km: 4
Pause

You will be called again with this:

Observation: miles_to_km: 4

You then output:

Answer: 4 miles is 6.43738 km

""".strip()

Agent

In [39]:
class Agent:
  def __init__(self, system=""):
    self.system = system
    self.messages = []
    if self.system:
      self.messages.append({"role": "system", "content": system})

  def __call__(self, message):
    self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result

  def execute(self):
    response = client.responses.create(
        model="gpt-6-astra",
        input = self.messages
    )
    return response.output_text

Step by Step test

In [40]:
agent01 = Agent(prompt)

In [41]:
result = agent01("20 celsius to fahrenheit, 20 miles to km")
print(result)

Thought: I’ll convert the temperature first, then the distance.
Action: celsius_to_fahrenheit: 20
Pause


In [42]:
next_prompt = f"Observation: {result}"
agent01(next_prompt)

'Thought: I’ll now convert 20 miles to kilometers.\nAction: miles_to_km: 20\nPause'

In [43]:
next_prompt = f"Observation: {result}"
print(agent01(next_prompt))

Answer:
- 20°C = **68°F**
- 20 miles = **32.18688 km**


Add Loop

In [47]:
action_re = re.compile('^Action: (\w+): (.*)$')

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_1312/1336279184.py:1: SyntaxWarning: invalid escape sequence '\w'
  action_re = re.compile('^Action: (\w+): (.*)$')


In [48]:
def query(question, max_turns = 10):
  i = 0
  bot = Agent(prompt)
  next_prompt = question
  while i < max_turns:
    i += 1
    result = bot(next_prompt)
    print(result)
    actions = [
        action_re.match(a)
        for a in result.split('\n')
        if action_re.match(a)
    ]
    if actions:
      action, action_input = actions[0].groups()
      if action not in known_actions:
        raise Exception(f"Unknown action: {action}:{action_input}")
      print(f"--- running {action}:{action_input}")
      Observation = known_actions[action](action_input)
      print("Observation: ", Observation)
      next_prompt = f"Observation: {Observation}"
    else:
      return

Test

In [49]:
question = "20 miles to km and 30 celsius to fahrenheit"
query(question)

Thought: I’ll convert miles to kilometers and Celsius to Fahrenheit.
Action: miles_to_km: 20
Action: celsius_to_fahrenheit: 30
Pause
--- running miles_to_km:20
Observation:  32.1868
Thought: The distance conversion is complete; I’ll now convert the temperature.
Action: celsius_to_fahrenheit: 30
Pause
--- running celsius_to_fahrenheit:30
Observation:  86.0
Answer: 20 miles = 32.1868 km, and 30°C = 86°F.
